<a href="https://colab.research.google.com/github/thequantumbruh/LLMs/blob/main/Exploring_MoE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch.nn as nn
import torch

In [3]:
class gated_network (nn.Module):
  def __init__(self,in_dim, num_expert, k = 1):
    super.__init__()
    self.k = k
    self.in_dim = in_dim
    self.num_experts = num_expert
    self.gated_linear = nn.Linear (self.in_dim ,num_expert)


  def forward (self, x: torch.Tensor)->torch.Tensor:
    logits = self.gated_linear(x)
    size = x.size()
    top_k, k_indices = torch.topk(logits, self.k, dim =-1)
    probabilities = nn.Softmax(logits)
    full_weights = torch.zeros(size[0], self.num_experts)
    full_weights.scatter(probabilities,-1, k_indices)

    return full_weights, k_indices

In [4]:
class experts (nn.Module):
  def __init__ (self,d_in: int, hid_dim: int, d_out: int):
    super.__init__()
    self.FF_block1 = nn.Sequential(
        nn.Linear(d_in, hid_dim),
        nn.GELU(),
        nn.Linear(hid_dim,d_out),
        nn.Dropout(0.012)
    )

    self.FF_block2 = nn.Sequential(
        nn.Linear(d_in, hid_dim),
        nn.GELU(),
        nn.Linear(hid_dim,d_out)
    )
  def forward (self, x:torch.Tensor)->torch.Tensor:
    x = self.FF_block1(x)
    x = self.FF_block2(x)
    return x

In [5]:
#MOE Layer without Load balancing and Expert capacity Routing

class MOE (nn.Module):
  def __init__(self, num_expert: int, d_in: int, k: int, d_out: int, hidden_dim = 4):
    super().__init__()
    self.expert = nn.ModuleList(experts(d_in, hidden_dim, d_out) for _ in range(num_expert))
    self.num_expert = num_expert
    self.k = k
    self.gated_network = gated_network(d_in, k, self.num_expert)

  def forward(self, x:torch.Tensor)->torch.Tensor:
    batch_size, seq_len, emb_dim = x.size
    #gated_input = x.view(-1, emb_dim)
    x = x.view(-1, emb_dim)
    gate_weights, k_indices = self.gated_network(x)
    final_outputs = torch.zeros(seq_len, emb_dim)

    flat_top_k = k_indices.flatten()

    expert_in = x.repeat_interleave(self.k, dim = 0)

    expert_outputs = []


    for i in range(self.num_expert):
      idx = torch.where(i == flat_top_k)[0]

      if len(idx)>0:
        expert_output = (self.expert[i](expert_in[idx]))
        expert_outputs.append((idx, expert_output))

    for idx, output in expert_outputs:
      original_idx = idx//self.k


      #since weights from the gate router is 2d matrix of N by K then we have to access Nth and Kth position
      #we access the token position(Nth) and the expert position(Kth)
      weights = gate_weights[original_idx, flat_top_k[idx]].unsqueeze(1)

      gated_weights = weights * output


      final_outputs.index_add_(dim = 0,index = original_idx, source = gated_weights)

      final_output = final_outputs.view(batch_size, seq_len, emb_dim)

    return final_output

In [6]:
class pull_package ():
  def __init__ (self, link: str, tokenizer):
    self.tokenizer = tokenizer
    self.data_loader = torch.utils.data.DataLoader
    self.dataset = torch.utils.data.Dataset

  def data_pull_tokenize():




def training_func(epochs: int, model):
  for epoch in range (epochs):
    model.train()

    for batch, (x_train)


IndentationError: expected an indented block after function definition on line 7 (2038713195.py, line 12)

In [9]:
help(torch.utils.data.dataset)

Help on module torch.utils.data.dataset in torch.utils.data:

NAME
    torch.utils.data.dataset - # mypy: allow-untyped-defs

CLASSES
    typing.Generic(builtins.object)
        Dataset
            ConcatDataset
            IterableDataset(Dataset, collections.abc.Iterable, typing.Generic)
                ChainDataset
            StackDataset
            Subset
            TensorDataset

    class ChainDataset(IterableDataset)
     |  ChainDataset(datasets: Iterable[torch.utils.data.dataset.Dataset]) -> None
     |
     |  Dataset for chaining multiple :class:`IterableDataset` s.
     |
     |  This class is useful to assemble different existing dataset streams. The
     |  chaining operation is done on-the-fly, so concatenating large-scale
     |  datasets with this class will be efficient.
     |
     |  Args:
     |      datasets (iterable of IterableDataset): datasets to be chained together
     |
     |  Method resolution order:
     |      ChainDataset
     |      IterableDataset

In [8]:
data_loader = torch.utils.data.dataloader(data)

Help on module torch.utils.data.dataloader in torch.utils.data:

NAME
    torch.utils.data.dataloader - Definition of the DataLoader and associated iterators that subclass _BaseDataLoaderIter.

DESCRIPTION
    To support these two classes, in `./_utils` we define many utility methods and
    functions to be run in multiprocessing. E.g., the data loading worker loop is
    in `./_utils/worker.py`.

CLASSES
    typing.Generic(builtins.object)
        DataLoader

    class DataLoader(typing.Generic)
     |  DataLoader(
     |      dataset: 'Dataset[_T_co]',
     |      batch_size: 'int | None' = 1,
     |      shuffle: 'bool | None' = None,
     |      sampler: 'Sampler | Iterable | None' = None,
     |      batch_sampler: 'Sampler[list] | Iterable[list] | None' = None,
     |      num_workers: 'int' = 0,
     |      collate_fn: '_collate_fn_t | None' = None,
     |      pin_memory: 'bool' = False,
     |      drop_last: 'bool' = False,
     |      timeout: 'float' = 0,
     |      worker